# Week 4 — Task 1 & Task 2 Reports
**NLP Training — Project 2**

This notebook contains:
1. Task 1: ALBERT (A Lite BERT) Report
2. Task 2: LoRA and QLoRA Report


---
# Task 1: ALBERT (A Lite BERT)
### A Lightweight and Efficient Member of the BERT Family


## 1. Introduction and Motivation

BERT (Bidirectional Encoder Representations from Transformers) established a new paradigm for natural language understanding by pretraining deep bidirectional Transformer encoders on large text corpora and fine-tuning them on downstream tasks. However, the pursuit of ever-better performance led researchers to build increasingly large BERT variants, and this trend exposed a fundamental problem: simply making BERT deeper and wider does not scale well. Larger models require more GPU/TPU memory, take longer to train, and beyond a certain size can actually become harder to train effectively, sometimes even degrading in performance due to optimization difficulties.

ALBERT (A Lite BERT), introduced by Lan et al. in 2019, was designed specifically to address this scaling problem. Its central motivation is not simply to shrink BERT, but to redesign the parameter allocation of the architecture so that model capacity can grow without a proportional explosion in the number of parameters. The result is a model family that is dramatically smaller than BERT in parameter count, yet matches or exceeds BERT's performance on major language understanding benchmarks such as GLUE, SQuAD, and RACE.


## 2. Limitations of Standard BERT That ALBERT Addresses

**Memory limitations at scale:** As hidden layer size increases, the number of parameters in BERT grows very quickly because every one of the 12 (or 24) Transformer layers has its own independent set of weights. This makes very large BERT models memory-hungry and expensive to train.

**Longer training times:** More parameters mean more computation and communication overhead during distributed training, which slows down experimentation and increases hardware cost.

**Diminishing and even degrading returns from scaling:** The ALBERT authors observed that naively increasing BERT-Large's hidden size (e.g., to create a "BERT-xlarge") led to worse performance, not better, indicating that simple scaling runs into representational and optimization bottlenecks rather than a lack of capacity.

**Redundancy in the Next Sentence Prediction (NSP) task:** BERT's NSP pretraining objective conflates topic prediction and coherence prediction, and subsequent research showed it contributes little useful signal, sometimes even hurting downstream performance.


## 3. ALBERT Architecture Overview

At a high level, ALBERT keeps the same Transformer encoder backbone as BERT: multi-head self-attention followed by position-wise feed-forward layers, with residual connections and layer normalization, operating over WordPiece-tokenized input augmented with positional and segment embeddings. What changes is how parameters are allocated and reused across the network. ALBERT introduces three central design innovations: **factorized embedding parameterization**, **cross-layer parameter sharing**, and a new self-supervised pretraining task called **Sentence Order Prediction (SOP)**. Together these allow ALBERT to use far fewer parameters than BERT while preserving representational power.


## 4. Key Innovations

### 4.1 Factorized Embedding Parameterization

In standard BERT, the WordPiece embedding matrix has size V × H, where V is the vocabulary size (typically around 30,000) and H is the hidden size of the Transformer layers (e.g., 768 for BERT-Base). Because H is tied directly to the embedding dimension, increasing the hidden size for greater model capacity forces a proportional increase in the enormous embedding matrix as well, even though the embedding layer mainly needs to encode context-independent, relatively low-level information.

ALBERT decouples the size of the hidden layers from the size of the vocabulary embedding by factorizing the embedding matrix into two smaller matrices: first projecting one-hot vocabulary vectors into a lower-dimensional embedding space of size E (with E much smaller than H), and then projecting that E-dimensional embedding into the H-dimensional hidden space used by the Transformer layers. This reduces the embedding parameter count from O(V × H) to O(V × E + E × H), which is a substantial saving when H is large, since E can be kept small (for example, 128) regardless of how large H grows.

### 4.2 Cross-Layer Parameter Sharing

BERT allocates a separate, independently learned set of weights (attention weights and feed-forward weights) to every Transformer layer. ALBERT instead shares parameters across all layers, so effectively every layer reuses the same underlying weight matrices. In its default configuration, ALBERT shares both the attention parameters and the feed-forward network parameters across layers, meaning the total parameter count no longer scales with network depth.

This has two effects. First, it drastically reduces the parameter count, since adding more layers no longer multiplies the parameter budget. Second, the authors found that it has a regularizing effect, stabilizing the transition of parameters from layer to layer and making the network's learned representations oscillate less between layers than they do in standard BERT, which can aid optimization stability at greater depth.

### 4.3 Sentence Order Prediction (SOP)

BERT's Next Sentence Prediction (NSP) task asks the model to predict whether two given segments follow one another in the original text or are random unrelated segments. Follow-up research found that NSP is an easy task that can often be solved through topic detection alone, giving the model little incentive to learn genuine inter-sentence coherence, and NSP was shown to sometimes hurt performance compared to omitting it entirely.

ALBERT replaces NSP with Sentence Order Prediction (SOP). In SOP, both training examples are always two consecutive segments taken from the same document; the negative example is created by simply swapping the order of these two segments rather than sampling an unrelated segment from elsewhere. Because both positive and negative examples share the same topic, the model can no longer rely on topic cues and is instead forced to learn finer-grained discourse-level coherence, which the ALBERT authors show transfers better to downstream multi-sentence reasoning tasks.


## 5. How These Techniques Preserve Performance While Reducing Parameters

The combination of factorized embeddings and cross-layer sharing reduces ALBERT's parameter count by roughly an order of magnitude relative to BERT at a comparable hidden size (for example, ALBERT-Base has about 12 million parameters versus roughly 110 million for BERT-Base). Crucially, this parameter reduction does not come at the cost of representational capacity in the same way naive shrinking would, for two reasons. First, because the embedding and hidden dimensions are decoupled, ALBERT can still use a very large hidden size H (and therefore rich per-token representations) without paying for it in the embedding matrix. Second, because layers share weights, ALBERT can be made very deep or scaled up (as in ALBERT-xxlarge) without the parameter count spiraling out of control, allowing the freed-up parameter budget to instead be spent on width (larger H), which the authors found to be a more effective use of capacity than adding independent layers.

The net effect is that ALBERT models with far fewer total parameters than BERT can match or outperform BERT on benchmarks such as GLUE, SQuAD 1.1/2.0, and RACE, with ALBERT-xxlarge setting new state-of-the-art results at the time of its release despite having fewer parameters than BERT-Large.


## 6. Comparison: ALBERT vs. BERT

| Aspect | BERT (Base) | ALBERT (Base) |
|---|---|---|
| **Embedding parameterization** | Single large embedding matrix, size V × H | Factorized into V × E and E × H, with E << H |
| **Layer parameters** | Each of the 12 layers has its own independent set of weights | All layers share one set of weights (cross-layer parameter sharing) |
| **Total parameters** | ≈ 110 million | ≈ 12 million |
| **Next-sentence objective** | Next Sentence Prediction (NSP) | Sentence Order Prediction (SOP) |
| **Training efficiency** | Slower; more parameters to update per step | Faster convergence per parameter; lower memory lets larger batches be used |
| **Memory usage** | Higher; scales with full per-layer parameter count | Substantially lower; shared weights fit in far less memory |
| **Inference speed** | Baseline | Comparable or slightly slower per layer due to repeated computation over shared weights, despite fewer parameters |
| **Downstream performance** | Strong baseline (GLUE, SQuAD) | Matches or exceeds BERT-Base/Large on most benchmarks, especially at larger configurations (e.g., ALBERT-xxlarge) |
| **Scalability** | Degrades with very deep/wide models due to parameter explosion and training instability | Scales more gracefully to very large hidden sizes because parameters are shared and factorized |


## 7. Trade-offs and Practical Considerations

- **Training/inference speed:** Although ALBERT has far fewer parameters, cross-layer sharing means the same weights are applied repeatedly, so wall-clock inference time per layer is not proportionally faster than BERT; the primary gains are in memory footprint and the ability to train larger, more capable configurations within a fixed memory budget.
- **Communication overhead in distributed training:** Fewer parameters mean less data to synchronize across devices, which can meaningfully speed up distributed pretraining.
- **Best use case:** ALBERT is especially attractive when memory is the binding constraint (e.g., training very large models on limited hardware) or when a compact model is needed for deployment, while still requiring strong language understanding performance.


## 8. Conclusion

ALBERT demonstrates that the size of a Transformer language model, measured in parameters, is not the only lever for improving performance. By rethinking how parameters are allocated through factorized embedding parameterization and cross-layer parameter sharing, and by replacing the weak NSP objective with the more informative Sentence Order Prediction task, ALBERT achieves BERT-level or better language understanding with a fraction of the parameters. This makes it a compelling choice whenever memory efficiency, training scalability, or deployment footprint are important considerations, and it remains an influential example of parameter-efficient design in the broader Transformer literature.


---
# Task 2: LoRA and QLoRA
### Parameter-Efficient Fine-Tuning for Large Language Models


## 1. Motivation for Parameter-Efficient Fine-Tuning

Modern large language models (LLMs) contain anywhere from hundreds of millions to hundreds of billions of parameters. Full fine-tuning — updating every parameter of such a model for a downstream task — requires storing gradients and optimizer states for all of those parameters, which quickly becomes infeasible on typical hardware. For example, full fine-tuning of a model with tens of billions of parameters can require hundreds of gigabytes of GPU memory once weights, gradients, and optimizer states (e.g., Adam's two moment estimates) are accounted for. Full fine-tuning also produces a separate full-size copy of the model for every downstream task, which is costly to store and deploy.

Parameter-efficient fine-tuning (PEFT) methods address this by freezing the vast majority of a pretrained model's weights and training only a small number of additional or selected parameters. This dramatically reduces memory and storage costs, enables fine-tuning on modest hardware, and allows many task-specific "adapters" to share a single frozen base model. LoRA and QLoRA are two of the most widely used PEFT techniques.


## 2. LoRA: Low-Rank Adaptation

### 2.1 Core Idea

LoRA is based on the observation that the weight updates needed to adapt a pretrained model to a new task tend to have a low "intrinsic rank" — that is, the change in weights can be well approximated by a low-rank matrix, even though the original weight matrix itself is high-rank. Instead of updating a weight matrix W directly, LoRA freezes W and learns a separate low-rank decomposition of its update.

### 2.2 Mathematical Formulation

For a pretrained weight matrix W with dimensions d × k, LoRA represents the update ΔW as the product of two much smaller matrices, A and B, such that ΔW = B·A, where A has dimensions r × k and B has dimensions d × r, and the rank r is chosen to be much smaller than d and k (commonly r is 4, 8, 16, or 32). The modified forward pass becomes:

$$h = W \cdot x + \Delta W \cdot x = W \cdot x + B \cdot A \cdot x$$

During training, W remains frozen and only A and B are updated. A is typically initialized with small random (Gaussian) values and B is initialized to zero, so that ΔW is zero at the start of training and the fine-tuned model initially behaves exactly like the pretrained model. A scaling factor (α divided by r) is usually applied to ΔW to control the magnitude of the adaptation relative to the rank chosen.

### 2.3 Architectural Placement

LoRA is most commonly applied to the weight matrices inside the self-attention layers of a Transformer, particularly the **query** and **value** projection matrices, since these were found empirically to give the best trade-off between parameter efficiency and downstream performance. It can, however, be applied to any linear layer, including key and output projections or the feed-forward layers, at the cost of more trainable parameters.

Because ΔW = B·A can be computed once training is complete and simply added into W, LoRA introduces no additional inference latency: the adapted weights can be merged back into the original weight matrix, so the deployed model has exactly the same architecture and speed as the original.


## 3. QLoRA: Quantized Low-Rank Adaptation

QLoRA extends LoRA by combining it with aggressive quantization of the frozen base model, enabling fine-tuning of much larger models on hardware with limited GPU memory. The base model's weights are stored in 4-bit precision rather than the usual 16-bit, while the LoRA adapters (A and B) are still trained in higher precision. QLoRA introduces three key technical components on top of standard LoRA:

### 3.1 4-bit Quantization

Instead of storing the frozen pretrained weights in 16-bit floating point, QLoRA stores them using only 4 bits per parameter. This alone reduces the memory needed to hold the base model by roughly a factor of four. During the forward and backward pass, the relevant 4-bit weights are dequantized back to a higher-precision format (such as BF16) just long enough to perform the matrix multiplication, then discarded, so the memory savings are preserved throughout training.

### 3.2 NF4 (4-bit NormalFloat)

A naive 4-bit quantization scheme spaces the 16 representable values evenly, which is a poor fit for neural network weights, since pretrained weights are typically distributed roughly as a zero-centered Gaussian rather than uniformly. NF4 is a quantization data type specifically designed for normally distributed data: it places the 16 quantization levels at positions that are information-theoretically optimal for a Gaussian distribution, so that each of the 4-bit codes carries as much information as possible about the original weight values. This lets QLoRA quantize weights down to 4 bits with much less loss of model quality than a generic 4-bit scheme.

### 3.3 Double Quantization

Quantization itself requires storing extra metadata (quantization constants) used to map the 4-bit codes back to real values, and these constants are normally stored at higher precision. Double quantization further compresses this overhead by quantizing the quantization constants themselves, reducing the average memory footprint per parameter by a small but meaningful additional amount (roughly an extra 0.4 bits per parameter on average) without noticeably affecting model accuracy.

### 3.4 Paged Optimizers

QLoRA also introduces paged optimizers, which use NVIDIA's unified memory feature to automatically move optimizer state between GPU and CPU memory when GPU memory would otherwise be exceeded, such as during a sudden spike in memory usage from long input sequences. This prevents out-of-memory crashes without requiring the user to manually manage memory placement, at the cost of a slight slowdown during the pages that are actually offloaded.


## 4. Comparison: LoRA vs. QLoRA

| Aspect | LoRA | QLoRA |
|---|---|---|
| **Base model precision** | 16-bit (FP16/BF16) | 4-bit (NF4), dequantized to 16-bit on the fly for computation |
| **Trainable parameters** | Low-rank adapter matrices A and B only (typically < 1% of base model parameters) | Same as LoRA — adapter matrices A and B only; base model remains frozen and quantized |
| **Memory usage (base weights)** | Full 16-bit weights kept in GPU memory | Roughly 4x smaller due to 4-bit storage; e.g., a 65B model fits on a single ~48GB GPU |
| **Optimizer state memory** | Standard, proportional to trainable adapter parameters only | Further reduced via paged optimizers that offload optimizer state to CPU memory under pressure |
| **Computational requirements** | Forward/backward pass in 16-bit throughout | Extra dequantization step (4-bit → 16-bit) before each pass, adding modest compute overhead |
| **Training speed** | Fast, minimal overhead vs. full fine-tuning at equivalent batch size | Slightly slower per step than LoRA, but enables much larger models/batch sizes on the same hardware |
| **Model performance** | Matches full fine-tuning quality on most tasks | Matches LoRA and full fine-tuning quality; negligible loss from 4-bit quantization (per QLoRA paper) |
| **Hardware requirements** | Requires enough GPU memory for the full 16-bit base model plus activations | Enables fine-tuning much larger models on consumer/single-GPU hardware |
| **Typical use case** | Fine-tuning small–mid-sized models, or when GPU memory is already sufficient | Fine-tuning very large models (tens of billions of parameters) on limited hardware |


## 5. Summary of Trade-offs

- **Memory vs. speed:** QLoRA trades a modest amount of training speed (due to on-the-fly dequantization) for a large reduction in memory footprint, making it the better choice when GPU memory, not compute time, is the binding constraint.
- **Accuracy:** Both methods have been shown empirically to closely match full fine-tuning performance on a wide range of tasks; QLoRA's use of NF4 and double quantization specifically was designed to make 4-bit quantization nearly lossless for this purpose.
- **When to choose LoRA:** If the base model already fits comfortably in available GPU memory in 16-bit precision, plain LoRA is simpler and slightly faster, with no quantization overhead.
- **When to choose QLoRA:** If the base model is too large to fit in available GPU memory at 16-bit precision — which is common for large open-source LLMs on single-GPU or consumer hardware — QLoRA makes fine-tuning feasible by cutting the base model's memory footprint roughly fourfold while keeping adapter training in higher precision.


## 6. Conclusion

LoRA and QLoRA both address the core problem of parameter-efficient fine-tuning by learning small, low-rank weight updates while keeping the pretrained base model frozen. LoRA achieves this through low-rank decomposition of weight updates alone, while QLoRA builds on this by additionally quantizing the frozen base model to 4 bits using NF4 and double quantization, and by using paged optimizers to manage memory spikes. Together, these techniques make it possible to fine-tune large, high-performing language models on hardware that would otherwise be far too memory-constrained for full fine-tuning, with minimal loss in downstream task performance.
